# Hierarchical Data, the JSON Data Format, and APIs

In [ ]:
import pandas as pd

## Shows Data

First we'll work with the "Girls" shows JSON data from the reading.

In [ ]:
# Fetch data from a URL
import requests
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

import json
data_shows = response.json()

In [ ]:
df_shows = pd.json_normalize(data_shows)
df_shows

1\. Summarize the networks represented by these shows and the number of these shows that aired on each network.

In [ ]:
network_counts = df_shows["network.name"].value_counts(dropna=False)
network_counts

2\. Find the number of seasons for each show in the data. Do this two ways: one which uses `df_shows`, and another that first flattens `data_shows` to a different data frame.

In [ ]:
# Method 1: count the season objects stored in each show's row
season_counts_1 = pd.DataFrame({
    "Show": df_shows["name"],
    "Number of Seasons": df_shows["seasons"].apply(len)
})
display(season_counts_1)

# Method 2: flatten the seasons and count rows by show
df_seasons = pd.json_normalize(data_shows, record_path="seasons",
                               meta=["name"], meta_prefix="show.")
season_counts_2 = (df_seasons.groupby("show.name").size()
                   .reset_index(name="Number of Seasons"))
season_counts_2

3\. For each episode, find the length (number of characters) of the title. Then create summaries to answer: Which show tends to have the longest episode titles? The shortest?

In [ ]:
episode_rows = []
for show in data_shows:
    for season in show["seasons"]:
        for episode in season["episodes"]:
            episode_rows.append({
                "Show": show["name"],
                "Episode Title": episode["name"]
            })

df_episodes = pd.DataFrame(episode_rows)
df_episodes["Title Length"] = df_episodes["Episode Title"].str.len()

title_summary = (df_episodes.groupby("Show")["Title Length"]
                 .agg(["count", "mean", "median", "min", "max"])
                 .sort_values("mean", ascending=False))
display(title_summary)

print("Longest titles on average:", title_summary["mean"].idxmax())
print("Shortest titles on average:", title_summary["mean"].idxmin())

4\. Do any cast members in the data set share a birthday with you? Who, and what show are they on? (If there isn't anyone, try a different day.)

In [ ]:
# The prompt allows choosing another date if needed.
# Here I use January 1.
birthday_month_day = "01-01"

cast_rows = []
for show in data_shows:
    for credit in show["cast"]:
        person = credit["person"]
        cast_rows.append({
            "Show": show["name"],
            "Cast Member": person["name"],
            "Birthday": person.get("birthday")
        })

df_cast = pd.DataFrame(cast_rows)
birthday_matches = df_cast[
    df_cast["Birthday"].fillna("").str.endswith(birthday_month_day)
]
birthday_matches

## TVMaze API

Now you will work with the [TVMaze API](http://www.tvmaze.com/api) from the reading. Use the API to request JSON data that you can use to answer the following questions.

1\. What was the longest show that aired in the U.S. on February 4, 2018?

_Hint:_ Use the ["Schedule" endpoint](http://www.tvmaze.com/api#schedule) to first get the data for all shows that aired on that date.

In [ ]:
schedule_url = "https://api.tvmaze.com/schedule"
params = {"country": "US", "date": "2018-02-04"}

schedule_response = requests.get(schedule_url, params=params)
schedule_response.raise_for_status()
schedule_data = schedule_response.json()
df_schedule = pd.json_normalize(schedule_data)

longest_show = (df_schedule[["show.name", "show.runtime"]]
                .drop_duplicates()
                .dropna(subset=["show.runtime"])
                .sort_values("show.runtime", ascending=False))

display(longest_show.head())
print("Longest show:", longest_show.iloc[0]["show.name"])
print("Runtime:", longest_show.iloc[0]["show.runtime"], "minutes")

2\. Among all shows that aired in the U.S. on Feburary 4, 2018, which non-voice actors appeared on more than one show? Note: some people are credited with multiple rows on the same show and thus appear as multiple rows in the data frame.

Hint: You will need to write a for loop to make multiple requests. Use the "show.id" from the data from part 1, and use the ["Shows" endpoint](http://www.tvmaze.com/api#show-cast) to get the cast of each show. Don't forget to stagger your requests, or you will be blocked by the website!

In [ ]:
import time

show_ids = df_schedule["show.id"].dropna().astype(int).unique()
cast_records = []

for show_id in show_ids:
    cast_response = requests.get(f"https://api.tvmaze.com/shows/{show_id}/cast")
    cast_response.raise_for_status()
    show_name = df_schedule.loc[df_schedule["show.id"] == show_id, "show.name"].iloc[0]

    for credit in cast_response.json():
        if credit.get("voice") is False:
            cast_records.append({
                "Show ID": show_id,
                "Show": show_name,
                "Person ID": credit["person"]["id"],
                "Person": credit["person"]["name"]
            })
    time.sleep(0.25)

df_schedule_cast = pd.DataFrame(cast_records)
unique_person_show = df_schedule_cast.drop_duplicates(["Person ID", "Show ID"])

multi_show_actors = (unique_person_show.groupby(["Person ID", "Person"])
                     .agg(Number_of_Shows=("Show ID", "nunique"),
                          Shows=("Show", lambda x: ", ".join(sorted(set(x)))))
                     .reset_index())

multi_show_actors = multi_show_actors[multi_show_actors["Number_of_Shows"] > 1]
multi_show_actors.sort_values(["Number_of_Shows", "Person"],
                              ascending=[False, True])

# Tasty API

[Tasty.co](http://tasty.co) is a website and app that offers food recipes. They have made these recipes available through a [a REST API](https://rapidapi.com/apidojo/api/tasty). However, unlike the TVMaze API, this one requires authentication.

Specifically, you will need to create an account and subscribe to the "Basic" (free) plant. You will then be provided with an API key (X-Rapid-API-Key) that will need to be supplied with every request you make. This is used to track and limit usage.

1. [create an account](https://rapidapi.com/apidojo/api/tasty) and subscribe to the "Basic" (free) plan
2. log in and copy the X-RapidAPI-Key, which is a long string of letters and digits
3. paste this key to replace "PUT-YOUR-KEY-HERE" in the `headers` below

If you did everything correctly, then running the cell below should return a JSON object containing all the tags recognized by the Tasty API.

In [ ]:
import requests

domain = "https://tasty.p.rapidapi.com"
endpoint = "tags/list"
url = f"{domain}/{endpoint}"

headers = {
    "X-RapidAPI-Key": "7f4d5c04bbmsh22304187e7c4ad7p18495djsn1da952e82dd4",
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

response = requests.get(url, headers=headers)
response.raise_for_status()
response.json()

You will need to pass these `headers` with every HTTP request to the API. The API key (X-RapidAPI-Key) is how the server keeps track of how many requests you have made.

Take a look at [the documentation](https://rapidapi.com/apidojo/api/tasty). We will use the recipes/list endpoint.

Make sure you are logged into the account you are created, and select the recipes/list endpoint from the menu at left. Notice that this brings up a form that you can fill in, which generates the corresponding code.  By default, it provides Node.js code; change this to Python with the Requests client.

1\. Search for recipes containing "daikon" (an Asian radish) and request the JSON data, and convert it to a Pandas data frame.

In [ ]:
endpoint = "recipes/list"
url = f"{domain}/{endpoint}"
querystring = {"from": "0", "size": "40", "q": "daikon"}

response_daikon = requests.get(url, headers=headers, params=querystring)
response_daikon.raise_for_status()
daikon_json = response_daikon.json()

df_daikon = pd.json_normalize(daikon_json["results"])
df_daikon

2\. How many recipes containing daikon are there? Which one is the cheapest per portion?

In [ ]:
print("Number of daikon recipes:", len(df_daikon))

df_daikon["price_per_portion"] = (
    df_daikon["price.total"] / df_daikon["num_servings"]
)

cheapest_daikon = (df_daikon[
    ["name", "price.total", "num_servings", "price_per_portion"]
].dropna(subset=["price_per_portion"])
 .sort_values("price_per_portion")
 .head(1))

cheapest_daikon

3\. Find recipes containing avocado and request the JSON data.

**Hint:** Note that there are hundreds of results, but the API only returns 20 results by default and only 40 results maximum, even if you specify the `size=` parameter, so you will need to use a `for` loop, incrementing the `from=` parameter. Be sure to respect the API's rate limits (or you may be blocked!)

**Note:** Recall from the example in the reading that we created an empty list `episodes = []` and then added the results of each request to it in the for loop with `episodes.extend(response.json())`. But note that in the recipes/list endpoint there are two keys: count and results. We only want the results so try instead `.extend(response.json()).get("results")`.

**Suggestion:** Try writing a loop that only makes 2 or 3 requests first so you can test that it's working correctly. Also, make sure you use separate cells for code that makes requests to the API versus processing of the results; you don't want to rerun the requests unless absolutely necessary!

In [ ]:
first_params = {"from": "0", "size": "40", "q": "avocado"}
first_response = requests.get(f"{domain}/recipes/list",
                              headers=headers, params=first_params)
first_response.raise_for_status()
first_json = first_response.json()

avocado_results = list(first_json.get("results", []))
total_avocado = first_json.get("count", len(avocado_results))

for start in range(40, total_avocado, 40):
    params = {"from": str(start), "size": "40", "q": "avocado"}
    response = requests.get(f"{domain}/recipes/list",
                            headers=headers, params=params)
    response.raise_for_status()
    avocado_results.extend(response.json().get("results", []))
    time.sleep(0.25)

df_avocado = pd.json_normalize(avocado_results)
print("API-reported results:", total_avocado)
print("Recipes collected:", len(df_avocado))
df_avocado.head()

4\. For the recipes containing avocado, compute the proportion of reviews that are positive. For the avocado recipes with over 500 reviews, which one has the highest proportion of positive reviews?


In [ ]:
df_avocado["total_reviews"] = (
    df_avocado["user_ratings.count_positive"].fillna(0) +
    df_avocado["user_ratings.count_negative"].fillna(0)
)

df_avocado["proportion_positive"] = (
    df_avocado["user_ratings.count_positive"] /
    df_avocado["total_reviews"]
)

over_500 = df_avocado[df_avocado["total_reviews"] > 500]

best_reviewed = (over_500[
    ["name", "user_ratings.count_positive",
     "user_ratings.count_negative", "total_reviews",
     "proportion_positive"]
].sort_values("proportion_positive", ascending=False).head(1))

best_reviewed

5\. Take the avocado JSON data from above (you do NOT need to read in the data from the REST API again). How many recipes are vegetarian? You should be able to identify this from the "tags" attribute.

Hint: Try using `json_normalize` with "tags" as the record path to flatten the data so that there is one row for each tag.

In [ ]:
df_avocado_tags = pd.json_normalize(
    avocado_results,
    record_path="tags",
    meta=["id", "name"],
    meta_prefix="recipe."
)

vegetarian_recipes = (df_avocado_tags[
    df_avocado_tags["name"].str.lower().eq("vegetarian")
][["recipe.id", "recipe.name"]].drop_duplicates())

print("Number of vegetarian avocado recipes:", len(vegetarian_recipes))
vegetarian_recipes